In [1]:
import torch

import cheetah

In [2]:
incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=100_000,
    sigma_x=torch.tensor(1e-3),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)
histogram_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="histogram",
)
kde_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="kde",
)
cic_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="cloud-in-cell",
)

In [3]:
%%timeit
histogram_screen.track(incoming)
reading = histogram_screen.reading

3.1 ms ± 319 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
%%timeit
kde_screen.track(incoming)
reading = kde_screen.reading

178 ms ± 3.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
cic_screen.track(incoming)
reading = cic_screen.reading

7.38 ms ± 40.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
vectorized_incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=10_000,
    sigma_x=torch.linspace(1e-3, 2e-3, 100),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)

In [7]:
%%timeit
kde_screen.track(vectorized_incoming)
reading = kde_screen.reading

1.91 s ± 5.29 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
cic_screen.track(vectorized_incoming)
reading = cic_screen.reading

49.2 ms ± 554 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
mps_incoming = incoming.to(device="mps", dtype=torch.float32)
mps_histogram_screen = histogram_screen.to(device="mps", dtype=torch.float32)
mps_kde_screen = kde_screen.to(device="mps", dtype=torch.float32)
mps_cic_screen = cic_screen.to(device="mps", dtype=torch.float32)

In [10]:
# %%timeit
# mps_histogram_screen.track(mps_incoming)
# reading = mps_histogram_screen.reading

In [11]:
%%timeit
mps_kde_screen.track(mps_incoming)
reading = mps_kde_screen.reading

90.8 ms ± 1.11 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
%%timeit
mps_cic_screen.track(mps_incoming)
reading = mps_cic_screen.reading

14 ms ± 1.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
mps_vectorized_incoming = vectorized_incoming.to(device="mps", dtype=torch.float32)

In [14]:
%%timeit
kde_screen.track(vectorized_incoming)
reading = kde_screen.reading

789 ms ± 1.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [15]:
%%timeit
cic_screen.track(vectorized_incoming)
reading = cic_screen.reading

26.2 ms ± 1.04 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
